In [4]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 0.13
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 5 v2 — Clasificación (corregido)
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# XGBoost y LightGBM (opcionales)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR      = 'ml_normalized_grouped'
OUTPUT_DIR     = 'ml_results_classification'
MAX_OCCUPANCY  = 30  # Coherente con regresión
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 1. CARGAR DATOS
# ==========================================
print("\n1. LOADING DATA...")

X_train_norm = pd.read_excel(os.path.join(INPUT_DIR, 'X_train_normalised.xlsx')).values
X_test_norm  = pd.read_excel(os.path.join(INPUT_DIR, 'X_test_normalised.xlsx')).values
X_train_raw  = pd.read_excel(os.path.join(INPUT_DIR, 'X_train.xlsx')).values
X_test_raw   = pd.read_excel(os.path.join(INPUT_DIR, 'X_test.xlsx')).values
y_train_reg  = pd.read_excel(os.path.join(INPUT_DIR, 'y_train.xlsx')).values.ravel()
y_test_reg   = pd.read_excel(os.path.join(INPUT_DIR, 'y_test.xlsx')).values.ravel()

with open(os.path.join(INPUT_DIR, 'selected_features.pkl'), 'rb') as f:
    selected_features = pickle.load(f)

# ==========================================
# 1b. CONVERSIÓN A CATEGORÍAS
# ==========================================
def to_category(y):
    return np.select(
        [y == 0, (y >= 1) & (y <= MAX_OCCUPANCY), y > MAX_OCCUPANCY],
        [0, 1, 2]
    )

y_train = to_category(y_train_reg)
y_test  = to_category(y_test_reg)

class_names = ['Empty(0)', f'Low(1-{MAX_OCCUPANCY})', f'High({MAX_OCCUPANCY+1}+)']

print(f"   X_train: {X_train_raw.shape} | X_test: {X_test_raw.shape}")
print(f"   y_train: {y_train.shape} | y_test: {y_test.shape}")

for cat, name in enumerate(class_names):
    print(f"      {name}: train={np.sum(y_train==cat)}, test={np.sum(y_test==cat)}")

# ==========================================
# 2. MODELOS
# ==========================================
print("\n2. DEFINING MODELS...")

needs_scaling = {'Logistic Regression', 'SVM'}

models = {
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'SVM':                 SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42)
}

if HAS_XGB:
    models['XGBoost'] = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbosity=0
    )

if HAS_LGB:
    models['LightGBM'] = LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbose=-1
    )

cv_splitter = TimeSeriesSplit(n_splits=3)
print(f"   Modelos: {len(models)}")

# ==========================================
# 3. ENTRENAR CON VALIDACIÓN CRUZADA PRIMERO
# ==========================================
print("\n3. CROSS-VALIDATION...")
print("-"*60)

cv_results = {}

for name, model in models.items():
    X_tr = X_train_norm if name in needs_scaling else X_train_raw
    cv = cross_val_score(model, X_tr, y_train, cv=cv_splitter, scoring='f1_weighted')
    cv_results[name] = {'CV_F1_mean': cv.mean(), 'CV_F1_std': cv.std()}
    print(f"   {name:<25s} CV F1: {cv.mean():.4f} ± {cv.std():.4f}")

# ==========================================
# 4. ENTRENAR Y EVALUAR EN TEST
# ==========================================
print("\n4. TRAINING...")
print("-"*60)

results        = {}
trained_models = {}
predictions    = {}
confusion_matrices = {}

for name, model in models.items():
    X_tr = X_train_norm if name in needs_scaling else X_train_raw
    X_te = X_test_norm if name in needs_scaling else X_test_raw

    print(f"\n   [{name}] ({'norm' if name in needs_scaling else 'raw'})")

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    results[name] = {
        'Accuracy':  round(acc, 4),
        'Precision': round(prec, 4),
        'Recall':    round(rec, 4),
        'F1':        round(f1, 4),
        'CV_F1':     cv_results[name]['CV_F1_mean'],
        'CV_std':    cv_results[name]['CV_F1_std']
    }

    trained_models[name] = model
    predictions[name]    = y_pred
    confusion_matrices[name] = confusion_matrix(y_test, y_pred)

    print(f"      Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")

# ==========================================
# 5. MEJOR MODELO
# ==========================================
print("\n" + "="*60)
print("5. BEST MODEL")
print("="*60)

df_summary = pd.DataFrame(results).T.sort_values('F1', ascending=False)
best = df_summary.index[0]

print(f"\n   BEST: {best}")
print(f"   Accuracy:  {results[best]['Accuracy']*100:.1f}%")
print(f"   F1:        {results[best]['F1']*100:.1f}%")
print(f"   CV F1:     {results[best]['CV_F1']*100:.1f}%")

# Matriz de confusión del mejor
cm = confusion_matrices[best]
print(f"\n   Confusion Matrix:\n{cm}")

# ==========================================
# 6. GUARDAR
# ==========================================
print("\n6. SAVING...")

with open(os.path.join(OUTPUT_DIR, 'best_model.pkl'), 'wb') as f:
    pickle.dump(trained_models[best], f)

with open(os.path.join(OUTPUT_DIR, 'all_models.pkl'), 'wb') as f:
    pickle.dump(trained_models, f)

df_preds = pd.DataFrame({
    'Actual':    y_test,
    'Predicted': predictions[best],
    'Correct':   (y_test == predictions[best]).astype(int)
})
df_preds.to_excel(os.path.join(OUTPUT_DIR, 'predictions.xlsx'), index=False)

df_summary.to_excel(os.path.join(OUTPUT_DIR, 'models_summary.xlsx'))

df_all = pd.DataFrame({'Actual': y_test})
for name, y_pred in predictions.items():
    df_all[name] = y_pred
df_all.to_excel(os.path.join(OUTPUT_DIR, 'all_predictions.xlsx'), index=False)

# Guardar todas las matrices de confusión
np.save(os.path.join(OUTPUT_DIR, 'confusion_matrices.npy'), confusion_matrices)

# Guardar importancias de todos los modelos de árboles
for name, model in trained_models.items():
    if hasattr(model, 'feature_importances_'):
        df_imp = pd.DataFrame({
            'Feature': selected_features,
            'Importance': model.feature_importances_
        }).sort_values('Importance', ascending=False)
        df_imp.to_excel(os.path.join(OUTPUT_DIR, f'feature_importance_{name}.xlsx'), index=False)

# Reporte de clasificación
report_dict = classification_report(y_test, predictions[best], target_names=class_names, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()
df_report.to_excel(os.path.join(OUTPUT_DIR, 'classification_report.xlsx'))

with open(os.path.join(OUTPUT_DIR, 'results.pkl'), 'wb') as f:
    pickle.dump(results, f)
with open(os.path.join(OUTPUT_DIR, 'predictions.pkl'), 'wb') as f:
    pickle.dump(predictions, f)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train)

print(f"\n   Guardado en {OUTPUT_DIR}/:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    print(f"      {fname} ({size:.0f} KB)")

print("\n" + "="*60)
print("PASO 5 CLASIFICACIÓN COMPLETADO")
print("="*60)


1. LOADING DATA...
   X_train: (58, 10) | X_test: (25, 10)
   y_train: (58,) | y_test: (25,)
      Empty(0): train=22, test=8
      Low(1-30): train=15, test=12
      High(31+): train=21, test=5

2. DEFINING MODELS...
   Modelos: 7

3. CROSS-VALIDATION...
------------------------------------------------------------
   Random Forest             CV F1: 0.8452 ± 0.0776
   Gradient Boosting         CV F1: 0.8612 ± 0.0961
   Logistic Regression       CV F1: 0.8884 ± 0.0577
   SVM                       CV F1: 0.8884 ± 0.0276
   Decision Tree             CV F1: 0.7809 ± 0.0581
   XGBoost                   CV F1: 0.8895 ± 0.1090
   LightGBM                  CV F1: 0.3767 ± 0.3900

4. TRAINING...
------------------------------------------------------------

   [Random Forest] (raw)
      Acc: 0.8400 | Prec: 0.8498 | Rec: 0.8400 | F1: 0.8394

   [Gradient Boosting] (raw)
      Acc: 0.7200 | Prec: 0.7261 | Rec: 0.7200 | F1: 0.7095

   [Logistic Regression] (norm)
      Acc: 0.7600 | Prec: 0.8111